# Solver callbacks

> Per-iteration recorders of the solvers (costs, elites, distribution, gradients), as in stable-worldmodel.

In [ ]:
#| default_exp planning.solver.callbacks

In [ ]:
#| hide
from nbdev.showdoc import *

Pass `callbacks=[...]` to a solver that reports to them: `CEMSolver`, `ICEMSolver`, `GradientSolver`,
`CategoricalCEMSolver` and `CategoricalMPPISolver` (the ones of stable-worldmodel). Each solve resets them, every env
batch starts a new row and every iteration adds a value; `solve` returns their `history` (batches x
iterations) under `outputs['callbacks'][cb.output_key]`.

A callback computes one value per env and reduces it over the envs (`reduction` `'mean'`, `'sum'` or
`'none'`: the per-env list). The state the solvers pass follows the planning mode: costs are joint
`(B, S)`, or per agent `(B, S, A)` in `'per_agent'` mode, where the per-env values are per agent
too (`'none'` gives `[[agent values] per env]`). Candidates and actions carry the agent axis,
`(B, S, H, A, action_dim)`; the recorders reduce every axis after the ones they keep, so they apply to
both layouts.

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from typing import Any, Literal

import torch

Reduction = Literal['mean', 'sum', 'none']

In [ ]:
#| export
class Callback:
    """
    Base of the solver callbacks: `compute` returns a per-env metric reduced by
    `self._reduce`. `history` is `list[list[Any]]` (batches x steps).
    """
    name: str | None = None

    def __init__(self, reduction: Reduction = 'mean'):
        if reduction not in ('mean', 'sum', 'none'):
            raise ValueError(f"reduction must be 'mean', 'sum', or 'none', got {reduction!r}")
        self.reduction = reduction
        self.history: list[list[Any]] = []
        self._current: list[Any] = []

    def _reduce(self, x: torch.Tensor) -> float | list:
        x = x.detach()
        if self.reduction == 'mean':
            return x.mean().item()
        if self.reduction == 'sum':
            return x.sum().item()
        return x.cpu().tolist()

    def reset(self) -> None:
        self.history, self._current = [], []

    def start_batch(self) -> None:
        if self._current:
            self.history.append(self._current)
        self._current = []

    def end_solve(self) -> None:
        if self._current:
            self.history.append(self._current)
            self._current = []

    def __call__(self, **state: Any) -> None:
        value = self.compute(**state)
        if value is not None:
            self._current.append(value)

    def compute(self, **state: Any) -> Any:
        raise NotImplementedError

    @property
    def output_key(self) -> str:
        return self.name or self.__class__.__name__


class BestCostRecorder(Callback):
    "Minimum cost over the candidates, per env (per agent in 'per_agent' mode)."
    def compute(self, **state: Any) -> float | list:
        return self._reduce(state['costs'].min(dim=1).values)


class MeanCostRecorder(Callback):
    "Mean cost over the candidates, per env (per agent in 'per_agent' mode)."
    def compute(self, **state: Any) -> float | list:
        return self._reduce(state['costs'].mean(dim=1))

### CEM and iCEM

In [ ]:
#| export
class EliteCostRecorder(Callback):
    "Elite cost statistics (mean, min, max) per env."
    def compute(self, **state: Any) -> dict[str, float | list]:
        v = state['topk_vals'].detach()
        return {'mean': self._reduce(v.mean(dim=1)), 'min': self._reduce(v.min(dim=1).values),
                'max': self._reduce(v.max(dim=1).values)}


class VarNormRecorder(Callback):
    "Mean variance of the action distribution, per env."
    def compute(self, **state: Any) -> float | list:
        return self._reduce(state['var'].detach().flatten(1).mean(dim=-1))


class MeanShiftRecorder(Callback):
    "L2 distance between consecutive means of the distribution, per env."
    def compute(self, **state: Any) -> float | list | None:
        prev_mean = state.get('prev_mean')
        if prev_mean is None:
            return None
        return self._reduce((state['mean'] - prev_mean).detach().flatten(1).norm(dim=-1))


class EliteSpreadRecorder(Callback):
    "Standard deviation within the elites (their diversity), per env."
    def compute(self, **state: Any) -> float | list:
        return self._reduce(state['topk_candidates'].detach().std(dim=1).flatten(1).mean(dim=-1))

### Gradient descent

In [ ]:
#| export
class GradNormRecorder(Callback):
    """
    L2 norm of the action gradient per env (mean over the restarts). With `per_step=True`,
    one norm per horizon step (a list of length H), taken over everything after the horizon axis
    (the agents and action dimensions of `(B, N, H, A, D)` actions).
    """
    def __init__(self, reduction: Reduction = 'mean', per_step: bool = False):
        super().__init__(reduction=reduction)
        self.per_step = per_step

    def compute(self, **state: Any) -> Any:
        params: torch.Tensor = state['params']
        g = params.grad
        if self.per_step:
            H = params.shape[2]
            if g is None:
                return [0.0] * H
            per_env_per_step = g.detach().flatten(3).norm(dim=-1).mean(dim=1)      # (B, H)
            return [self._reduce(per_env_per_step[:, h]) for h in range(H)]
        if g is None:
            return 0.0
        return self._reduce(g.detach().flatten(2).norm(dim=-1).mean(dim=-1))


class ActionNormRecorder(Callback):
    "L2 norm of the actions per env (mean over the restarts)."
    def compute(self, **state: Any) -> float | list:
        return self._reduce(state['params'].detach().flatten(2).norm(dim=-1).mean(dim=-1))

### Tests

In [ ]:
from fastcore.test import test_fail

costs = torch.tensor([[3.0, 1.0, 2.0], [0.5, 4.0, 1.5]])           # (B, S) joint
cb = BestCostRecorder(); cb.reset(); cb.start_batch(); cb(step=0, costs=costs); cb.end_solve()
assert cb.history == [[0.75]]
per_agent = torch.stack([costs, costs * 2], dim=-1)                 # (B, S, A)
cb = BestCostRecorder(reduction='none'); cb.reset(); cb.start_batch(); cb(step=0, costs=per_agent); cb.end_solve()
assert cb.history == [[[[1.0, 2.0], [0.5, 1.0]]]]                   # per env, per agent
test_fail(lambda: MeanCostRecorder(reduction='median'), contains='reduction')

# gradient norms per horizon step with the agent axis: (B, N, H, A, D)
p = torch.randn(2, 3, 4, 2, 5, requires_grad=True)
(p ** 2).sum().backward()
cb = GradNormRecorder(per_step=True); cb.reset(); cb.start_batch(); cb(step=0, params=p); cb.end_solve()
expected = (2 * p.detach()).flatten(3).norm(dim=-1).mean(dim=1).mean(dim=0)
assert len(cb.history[0][0]) == 4 and torch.allclose(torch.tensor(cb.history[0][0]), expected, rtol=1e-5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()